# AI Services MS2 — Supplementary Case Study
## Courier Delivery Delay Risk Prediction | COMPLETE REFERENCE

**60 marks · Exactly 20 test cases.** Use `re` to extract structured shipment information and Scikit-learn `ColumnTransformer` for shared preprocessing. All CSV files must be beside this notebook.

### Imports and feature groups
These are shared across all assessment functions. Source filenames remain simple; no folder creation is required.

In [1]:
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score, f1_score

TARGET = "late_delivery"
RAW_NUMERIC = [
    "items_count", "driver_experience_years", "hub_queue_minutes",
    "prior_attempts", "promised_hours"
]
ENGINEERED_NUMERIC = ["distance_km", "parcel_weight_kg", "booking_hour", "is_weekend"]
NUMERIC_FEATURES = RAW_NUMERIC + ENGINEERED_NUMERIC
RAW_CATEGORICAL = ["service_type", "weather_condition", "hub_status"]
CATEGORICAL_FEATURES = RAW_CATEGORICAL + ["route_zone"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

### Regex helper
**`extract_note_features`** — Extract distance in kilometres, weight in kilograms, and route zone from a semi-structured note. Missing fields stay NaN or unknown.

In [2]:
def extract_note_features(note: object) -> tuple[float, float, str]:
    """Extract distance (km), weight (kg), and route zone from semi-structured dispatch notes."""
    text = "" if pd.isna(note) else str(note)
    distance = re.search(r"\bDIST\s*[:=]\s*(\d+(?:\.\d+)?)\s*km\b", text, flags=re.I)
    weight = re.search(r"\bWT\s*[:=]\s*(\d+(?:\.\d+)?)\s*kg\b", text, flags=re.I)
    zone = re.search(r"\bZONE\s*[:=]\s*(metro|regional|remote)\b", text, flags=re.I)
    return (
        float(distance.group(1)) if distance else np.nan,
        float(weight.group(1)) if weight else np.nan,
        zone.group(1).lower() if zone else "unknown",
    )

### Common data cleaning
**`load_data`** — Read CSV, remove duplicate shipment IDs, discard invalid timestamps, normalize category values, coerce numeric fields, and engineer regex/calendar features.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename)
    df = df.drop_duplicates("shipment_id", keep="first").copy()

    timestamps = pd.to_datetime( df["booked_at"], format="mixed",errors="coerce",dayfirst=True )

    df = df.loc[timestamps.notna()].copy()

    df["booking_hour"] = timestamps.loc[df.index].dt.hour
    df["is_weekend"] = ( timestamps.loc[df.index].dt.dayofweek >= 5 ).astype(int)

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

    for col in RAW_CATEGORICAL:
        df[col] = (
            df[col].astype("string").str.strip().str.lower()
            .fillna("unknown").replace("", "unknown").astype(str)
        )

    note_features = df["dispatch_note"].apply(extract_note_features)
    df["distance_km"] = note_features.map(lambda values: values[0])
    df["parcel_weight_kg"] = note_features.map(lambda values: values[1])
    df["route_zone"] = note_features.map(lambda values: values[2])
    
    return df

In [23]:
df=load_data("delivery_train.csv")

### Reusable preprocessing + estimator
**`build_pipeline`** — Use ColumnTransformer for training-fitted numeric median imputation/scaling and categorical one-hot encoding with unknown-category handling.

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    
    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("category", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])

    return Pipeline([("preprocessor", preprocessing), ("model", algorithm)])

### Prepare labeled training and test features
**`prepare_datasets`** — Apply common cleaning to both files; map no/yes target to 0/1; remove invalid labels; return train/test features and target series.

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    train_df = load_data(train_file)
    test_df = load_data(test_file)

    for df in (train_df, test_df):
        labels = df[TARGET].astype("string").str.strip().str.lower()
        df[TARGET] = labels.map({"no": 0, "yes": 1})
        df.dropna(subset=[TARGET], inplace=True)
        df[TARGET] = df[TARGET].astype(int)
        
    return train_df[FEATURES], test_df[FEATURES], train_df[TARGET], test_df[TARGET]

### Fit two candidate classifiers
**`train_models`** — Train separately fitted complete pipelines for Decision Tree and Random Forest with the required parameters.

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    models = {
        "decision_tree": DecisionTreeClassifier(
            max_depth=8, min_samples_leaf=4, class_weight="balanced", random_state=42
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=120, max_depth=10, min_samples_leaf=2,
            class_weight="balanced", random_state=42, n_jobs=-1
        ),
    }
    
    trained_models = {}
    for name, algorithm in models.items():
        pipeline = build_pipeline(algorithm)
        pipeline.fit(X_train, y_train)
        trained_models[name] = pipeline
    return trained_models

### Evaluate and compare
**`evaluate_models`** — Calculate balanced accuracy, precision, recall and F1 on the held-back data; rank by balanced accuracy.

In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    results = []
    for name, pipeline in trained_models.items():
        predictions = pipeline.predict(X_test)
        results.append({
            "model_name": name,
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions),
            "recall": recall_score(y_test, predictions),
            "f1": f1_score(y_test, predictions),
        })
    comparison = pd.DataFrame(results)
    comparison = comparison.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
    return comparison

### Select existing fitted model
**`select_best_model`** — Choose first model from sorted comparison and return its existing fitted pipeline.

In [8]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    best_name = comparison.iloc[0]["model_name"]
    return best_name, trained_models[best_name]

### Generate predicted risk
**`predict_delay`** — Return shipment IDs, binary predictions and class-1 probability rounded to four decimals.

In [9]:
def predict_delay(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    labels = model.predict(df[FEATURES]).astype(int)
    positive_index = list(model.classes_).index(1)
    probabilities = model.predict_proba(df[FEATURES])[:, positive_index]
    return pd.DataFrame({
        "shipment_id": df["shipment_id"].to_numpy(),
        "predicted_delay": labels,
        "delay_probability": np.round(probabilities, 4),
    })

### Training workflow
**`train`** — Train, evaluate, select, save best_model.pkl and model_comparison.csv.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    X_train, X_test, y_train, y_test = prepare_datasets("delivery_train.csv", "delivery_test.csv")
    
    trained = train_models(X_train, y_train)
    comparison = evaluate_models(trained, X_test, y_test)
    best_name, best_model = select_best_model(comparison, trained)

    joblib.dump(best_model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    
    print("Selected model:", best_name)
    print(comparison.round(3).to_string(index=False))
    
    return best_name, comparison

### Prediction workflow
**`predict`** — Reload best_model.pkl, load delivery_predict.csv, create delivery_predictions.csv without retraining.

In [ ]:
def predict() -> pd.DataFrame:
    model = joblib.load("best_model.pkl")
    queued = load_data("delivery_predict.csv")

    result = predict_delay(model, queued)

    result.to_csv("delivery_predictions.csv", index=False)

    print("Predictions generated:", len(result))
    
    print(result.head().to_string(index=False))
    return result

### Execute training
Run this cell when you want to fit and save the winning model. This operation is not triggered merely by defining the functions.

In [12]:
train()

Selected model: random_forest
   model_name  balanced_accuracy  precision  recall    f1
random_forest              0.645      0.486   0.536 0.510
decision_tree              0.619      0.455   0.497 0.475


('random_forest',
       model_name  balanced_accuracy  precision    recall        f1
 0  random_forest           0.645190   0.486486  0.536424  0.510236
 1  decision_tree           0.618848   0.454545  0.496689  0.474684)

### Execute independent prediction
Run this cell after training. It reloads the persisted model rather than fitting a new model.

In [13]:
predict()

Predictions generated: 300
  shipment_id  predicted_delay  delay_probability
SHP-26-000302                0             0.4297
SHP-26-003421                0             0.4738
SHP-26-005110                0             0.4593
SHP-26-001061                0             0.4077
SHP-26-002566                0             0.3203


,shipment_id,predicted_delay,delay_probability
0,SHP-26-000302,0,0.4297
1,SHP-26-003421,0,0.4738
2,SHP-26-005110,0,0.4593
3,SHP-26-001061,0,0.4077
4,SHP-26-002566,0,0.3203
...,...,...,...
295,SHP-26-003093,0,0.4936
296,SHP-26-003773,0,0.2576
297,SHP-26-005192,1,0.6104
298,SHP-26-005227,1,0.6613
